# Milestone 6 — Optimization, calibration, and final evaluation

Use a small class-weight/depth search on training data. Select the highest validation average precision; investigate sigmoid calibration with training-only cross-validation. Accept calibration only if validation Brier score improves at least 5% without losing more than 0.01 average precision. Select the highest-precision threshold satisfying validation recall ≥ 90%. This is an application objective, not an industrial safety guarantee.

No resampling is used: class weighting and threshold adjustment are tested first. Grouped and stratified cross-validation operate only on training rows. The selected pipeline stays trained on the training partition; validation remains available for selection, without refitting after threshold choice.

In [1]:
from pathlib import Path
import nbformat
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
notebook = nbformat.read(PROJECT_ROOT / "notebooks/04_baseline_models.ipynb", as_version=4)
for cell in notebook.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "baseline_definitions", "exec"), globals())
from xgboost import XGBClassifier
records, manifest = load_partitions()
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")

In [2]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import log_loss
imbalance_ratio = float((y_train == 0).sum() / (y_train == 1).sum())
experiments = {
    "logistic_balanced": LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE),
    "forest_balanced": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=4, random_state=RANDOM_STATE),
    "xgboost_weighted": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, scale_pos_weight=imbalance_ratio, subsample=0.9, tree_method="hist", n_jobs=4, random_state=RANDOM_STATE),
    "xgboost_depth4": XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, scale_pos_weight=3.0, subsample=0.9, reg_lambda=2, tree_method="hist", n_jobs=4, random_state=RANDOM_STATE),
}
results = json.loads((METRICS_DIR / "model_comparison.json").read_text())
for name, estimator in experiments.items():
    candidate = train_pipeline(estimator, X_train, y_train)
    p = candidate.predict_proba(X_validation)[:, 1]
    results.append({"model": name, "partition": "validation", **evaluate_probabilities(y_validation, p)})
    joblib.dump(candidate, MODEL_DIR / f"{name}.joblib")
selected = max(results, key=lambda item: item["pr_auc"])
base_pipeline = joblib.load(MODEL_DIR / f"{selected['model']}.joblib")
base_probabilities = base_pipeline.predict_proba(X_validation)[:, 1]
(METRICS_DIR / "model_comparison.json").write_text(json.dumps(results, indent=2) + "\n")
display(pd.DataFrame(results).drop(columns="confusion_matrix").sort_values("pr_auc", ascending=False))
print("Selected by validation average precision:", selected["model"])

,model,partition,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
6,xgboost_depth4,validation,0.5,0.839286,0.691176,0.758065,0.716463,0.972567,0.804986,0.012475,21,0.308824,2000,68
2,xgboost,validation,0.5,0.893617,0.617647,0.730435,0.658307,0.965481,0.794102,0.013479,26,0.382353,2000,68
1,random_forest,validation,0.5,0.852941,0.426471,0.568627,0.473856,0.955795,0.745368,0.016188,39,0.573529,2000,68
5,xgboost_weighted,validation,0.5,0.410072,0.838235,0.550725,0.693431,0.963228,0.719723,0.032356,11,0.161765,2000,68
4,forest_balanced,validation,0.5,0.780488,0.470588,0.587156,0.511182,0.959091,0.686854,0.017748,36,0.529412,2000,68
0,logistic_regression,validation,0.5,0.727273,0.235294,0.355556,0.272109,0.873714,0.432080,0.024883,52,0.764706,2000,68
3,logistic_balanced,validation,0.5,0.150838,0.794118,0.253521,0.428571,0.881051,0.407168,0.116299,14,0.205882,2000,68


Selected by validation average precision: xgboost_depth4


## Training-only cross-validation

The grouped check holds out whole 1,000-row CSV blocks among training records. It is a robustness diagnostic, not a forward-time deployment simulation. No final test rows enter any fold.

In [3]:
groups = np.asarray(manifest["train"]) // 1000
cv_results = []
protocols = {
    "stratified": StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE).split(X_train, y_train),
    "csv_blocks": GroupKFold(n_splits=3).split(X_train, y_train, groups),
}
for protocol, folds in protocols.items():
    for fold, (fit_indices, check_indices) in enumerate(folds, start=1):
        candidate = clone(base_pipeline).fit(X_train.iloc[fit_indices], y_train.iloc[fit_indices])
        p = candidate.predict_proba(X_train.iloc[check_indices])[:, 1]
        cv_results.append({"protocol": protocol, "fold": fold, **evaluate_probabilities(y_train.iloc[check_indices], p)})
(METRICS_DIR / "cross_validation.json").write_text(json.dumps(cv_results, indent=2) + "\n")
display(pd.DataFrame(cv_results).drop(columns="confusion_matrix"))

,protocol,fold,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,stratified,1,0.5,0.746032,0.701493,0.723077,0.709970,0.975361,0.765392,0.014686,20,0.298507,2000,67
1,stratified,2,0.5,0.750000,0.573529,0.650000,0.601852,0.968975,0.758858,0.015118,29,0.426471,2000,68
2,stratified,3,0.5,0.728814,0.632353,0.677165,0.649547,0.967719,0.730608,0.015293,25,0.367647,2000,68
3,csv_blocks,1,0.5,0.511628,0.523810,0.517647,0.521327,0.957639,0.586068,0.016637,20,0.476190,1808,42
4,csv_blocks,2,0.5,0.724138,0.333333,0.456522,0.373665,0.884502,0.490637,0.037041,84,0.666667,2382,126
5,csv_blocks,3,0.5,0.680000,0.485714,0.566667,0.515152,0.947235,0.535130,0.013435,18,0.514286,1810,35


## Calibration and threshold selection

In [4]:
calibrated = CalibratedClassifierCV(clone(base_pipeline), method="sigmoid", cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), n_jobs=1)
calibrated.fit(X_train, y_train)
calibrated_probabilities = calibrated.predict_proba(X_validation)[:, 1]
base_metrics = evaluate_probabilities(y_validation, base_probabilities)
calibrated_metrics = evaluate_probabilities(y_validation, calibrated_probabilities)
use_calibration = bool(calibrated_metrics["brier_score"] <= 0.95 * base_metrics["brier_score"] and calibrated_metrics["pr_auc"] >= base_metrics["pr_auc"] - 0.01)
production_pipeline = calibrated if use_calibration else base_pipeline
validation_probabilities = calibrated_probabilities if use_calibration else base_probabilities
threshold = select_threshold(y_validation, validation_probabilities)
threshold_rows = [evaluate_probabilities(y_validation, validation_probabilities, float(t)) for t in sorted(set([0.05, 0.1, 0.2, 0.3, 0.5, 0.7, threshold]))]
calibration_results = {"uncalibrated": base_metrics, "sigmoid": calibrated_metrics, "selected": "sigmoid" if use_calibration else "uncalibrated", "acceptance_rule": "At least 5% lower validation Brier and at most 0.01 AP loss"}
(METRICS_DIR / "calibration.json").write_text(json.dumps(calibration_results, indent=2) + "\n")
(METRICS_DIR / "threshold_analysis.json").write_text(json.dumps(threshold_rows, indent=2) + "\n")
display(pd.DataFrame(threshold_rows).drop(columns="confusion_matrix"))
print("Calibration accepted:", use_calibration, "Decision threshold:", threshold)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for label, p in (("Uncalibrated", base_probabilities), ("Sigmoid", calibrated_probabilities)):
    observed, predicted = calibration_curve(y_validation, p, n_bins=8, strategy="quantile")
    axes[0].plot(predicted, observed, marker="o", label=label)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set(xlabel="Mean predicted probability", ylabel="Observed failure rate", title="Validation calibration")
axes[0].legend()
for metric in ("precision", "recall", "f1"):
    axes[1].plot([r["threshold"] for r in threshold_rows], [r[metric] for r in threshold_rows], marker="o", label=metric)
axes[1].axvline(threshold, color="gray", linestyle="--")
axes[1].set(xlabel="Decision threshold", ylabel="Validation score", title="Threshold trade-offs")
axes[1].legend()
fig.savefig(FIGURE_DIR / "model_calibration_thresholds.png", dpi=160)
display(fig)
plt.close(fig)

,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,0.050000,0.358382,0.911765,0.514523,0.696629,0.972567,0.804986,0.012475,6,0.088235,2000,68
1,0.085678,0.449275,0.911765,0.601942,0.756098,0.972567,0.804986,0.012475,6,0.088235,2000,68
2,0.100000,0.465116,0.882353,0.609137,0.748130,0.972567,0.804986,0.012475,8,0.117647,2000,68
3,0.200000,0.571429,0.823529,0.674699,0.756757,0.972567,0.804986,0.012475,12,0.176471,2000,68
4,0.300000,0.638554,0.779412,0.701987,0.746479,0.972567,0.804986,0.012475,15,0.220588,2000,68
5,0.500000,0.839286,0.691176,0.758065,0.716463,0.972567,0.804986,0.012475,21,0.308824,2000,68
6,0.700000,0.926829,0.558824,0.697248,0.607029,0.972567,0.804986,0.012475,30,0.441176,2000,68


Calibration accepted: False Decision threshold: 0.08567804843187332


<Figure size 1100x400 with 2 Axes>

## Freeze the selected production artifact before test evaluation

In [5]:
joblib.dump(production_pipeline, MODEL_DIR / "production.joblib")
joblib.dump(base_pipeline, MODEL_DIR / "explanation_model.joblib")
artifact_hash = hashlib.sha256((MODEL_DIR / "production.joblib").read_bytes()).hexdigest()
metadata = {
    "model_version": "ai4i-" + artifact_hash[:12],
    "selected_model": selected["model"],
    "calibration": calibration_results["selected"],
    "decision_threshold": threshold,
    "threshold_objective": "Maximum validation precision with recall at least 0.90",
    "risk_thresholds": {"warning": threshold * 0.5, "high_risk": threshold, "critical": threshold + (1 - threshold) * 0.6},
    "risk_policy": "Application demonstration thresholds, not industrial standards",
    "features": list(FEATURE_COLUMNS),
    "dataset_sha256": manifest["source_sha256"],
    "artifact_sha256": artifact_hash,
    "training_records": len(X_train),
    "validation_metrics": evaluate_probabilities(y_validation, validation_probabilities, threshold),
    "limitations": ["Synthetic data", "Failure at observation, no future horizon", "Full-data EDA before partitioning", "Ordered operating regimes", "Rare and inconsistent failure-type labels"],
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Model and threshold frozen:", metadata["model_version"])

Model and threshold frozen: ai4i-f3eb2ed3aaf9


## Single final test evaluation

Do not use these results to revise the selected pipeline or threshold. A recall objective on validation is not guaranteed to transfer to test data.

In [6]:
X_test, y_test = get_partition(records, manifest, "test")
test_probabilities = production_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_probabilities(y_test, test_probabilities, threshold)
precision, recall, pr_thresholds = precision_recall_curve(y_test, test_probabilities)
fpr, tpr, roc_thresholds = roc_curve(y_test, test_probabilities)
curves = {"precision": precision.tolist(), "recall": recall.tolist(), "fpr": fpr.tolist(), "tpr": tpr.tolist()}
final = {"model_version": metadata["model_version"], "partition": "test", "metrics": test_metrics, "curves": curves}
(METRICS_DIR / "final_evaluation.json").write_text(json.dumps(final, indent=2) + "\n")
display(pd.DataFrame([test_metrics]).drop(columns="confusion_matrix"))
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
axes[0].plot(recall, precision)
axes[0].axhline(float(y_test.mean()), color="gray", linestyle="--")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Test precision-recall")
axes[1].plot(fpr, tpr)
axes[1].plot([0,1],[0,1],linestyle="--",color="gray")
axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Test ROC")
matrix = np.array(test_metrics["confusion_matrix"])
axes[2].imshow(matrix,cmap="Blues")
for row in range(2):
    for col in range(2):
        axes[2].text(col,row,str(matrix[row,col]),ha="center",va="center",color="white" if matrix[row,col]>matrix.max()/2 else "black")
axes[2].set(xticks=[0,1],yticks=[0,1],xlabel="Predicted",ylabel="Actual",title="Test confusion matrix")
fig.savefig(FIGURE_DIR / "final_model_evaluation.png",dpi=160)
display(fig)
plt.close(fig)
np.testing.assert_allclose(joblib.load(MODEL_DIR / "production.joblib").predict_proba(X_test.iloc[:10]), production_pipeline.predict_proba(X_test.iloc[:10]))
print("Final pipeline round-trip verified; no model changes made after test evaluation.")

,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,0.085678,0.405594,0.852941,0.549763,0.698795,0.98056,0.770586,0.013441,10,0.147059,2000,68


<Figure size 1400x400 with 3 Axes>

Final pipeline round-trip verified; no model changes made after test evaluation.
